In [3]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.6/314.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.9/233.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10

In [ ]:
import os
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

In [1]:
from datasets import Dataset
import json

# 1️⃣ Load your JSONL file line by line
with open("linux_shell_finetune_100.jsonl", "r") as f:
    data = [json.loads(line) for line in f]  # Each line becomes a dict

# 2️⃣ Convert to a Hugging Face Dataset
# Wrap each example in the format "Human prompt + Assistant completion"
dataset = Dataset.from_list([
    {
        "text": f"### Human:\n{item['prompt']}\n\n### Assistant:\n{item['completion'].strip()}"
    }
    for item in data
])

# ✅ Dataset is now ready
print(dataset[0])  # check first example


{'text': "### Human:\nWhy is my laptop so slow?\n\n### Assistant:\nRun 'top' to check CPU usage. Identify processes using high CPU and memory."}


In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-medium-4k-instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.7: Fast Mistral patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.71G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/318 [00:00<?, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        'q_proj' , 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha = 64 * 2,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth'
)

Unsloth 2025.9.7 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [4]:
from trl import SFTTrainer, SFTConfig

In [5]:
import os
os.environ["WANDB_DISABLED"] = "true"

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = 'text',
    tokenizer = tokenizer,
    max_seq_length = 2048,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        num_train_epochs = 3,
        logging_steps = 1,
        output_dir = 'output',
        optim = 'adamw_8bit'
    )
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/106 [00:00<?, ? examples/s]

In [6]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 106 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 262,144,000 of 14,222,382,080 (1.84% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.567100
2,2.707100
3,2.626300
4,2.514400
5,2.490100
6,2.261000
7,1.876100
8,1.720000
9,1.694000
10,1.259600


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TrainOutput(global_step=60, training_loss=0.7497242209812005, metrics={'train_runtime': 507.943, 'train_samples_per_second': 0.945, 'train_steps_per_second': 0.118, 'total_flos': 4923299148595200.0, 'train_loss': 0.7497242209812005, 'epoch': 4.30188679245283})

In [2]:
!pip install protobuf==3.20.3 --quiet
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

In [9]:
model.save_pretrained_gguf('finetuned_model', tokenizer, quantization_method='q4_k_m', maximum_memory_usage=0.3)

Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 7.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 1.5 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


  0%|          | 0/40 [00:00<?, ?it/s]
We will save to Disk and not RAM now.
100%|██████████| 40/40 [09:30<00:00, 14.27s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving finetuned_model/pytorch_model-00001-of-00006.bin...
Unsloth: Saving finetuned_model/pytorch_model-00002-of-00006.bin...
Unsloth: Saving finetuned_model/pytorch_model-00003-of-00006.bin...
Unsloth: Saving finetuned_model/pytorch_model-00004-of-00006.bin...
Unsloth: Saving finetuned_model/pytorch_model-00005-of-00006.bin...
Unsloth: Saving finetuned_model/pytorch_model-00006-of-00006.bin...
Done.


Unsloth: Converting mistral model. Can use fast conversion = True.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at finetuned_model into f16 GGUF format.
The output location will be /content/finetuned_model/unsloth.F16.gguf
This might take 3 minutes...


TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [10]:
pip install protobuf==3.20.3 --force-reinstall

  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.


In [ ]:
!du -h --max-depth=1 | sort -hr

55G	.
53G	./finetuned_model
1.5G	./output
271M	./llama.cpp
55M	./sample_data
4.0M	./huggingface_tokenizers_cache
2.6M	./unsloth_compiled_cache
140K	./.config
8.0K	./_unsloth_sentencepiece_temp


In [1]:
# Remove old checkpoints except the last one
rm -rf ./finetuned_model/checkpoint-*

SyntaxError: invalid syntax (ipython-input-2891506674.py, line 2)

In [4]:
!rm -rf output/
!rm -rf ~/.cache/huggingface